# 04 Embeddings, semantic axes, decoding, BERTopic
Registered plan, Sections 7.3, 7.4 (effect size), and 7.6. Use a GPU runtime (Colab Pro+) for `Qwen/Qwen3-Embedding-8B`. API keys come from Colab Secrets: `GEMINI_API_KEY`, `VOYAGE_API_KEY`.

*Generated by `_build_notebooks.py`; edit that file, not this notebook.*

In [ ]:
# --- Setup -------------------------------------------------------------
# MODE = "simulated" runs on the synthetic data (stand-ins replace the GPU models
# and APIs). MODE = "real" runs on the de-identified data. Until
# Analysis/text_frozen/text_config.json has frozen = true, real mode processes
# only the development set (plan, Section 7.1).
#
# In Colab, the code is a fresh clone of the GitHub repository, so the freeze
# check can verify the commit. The data are read from, and the results written
# to, the copy of the repository in your Google Drive. Its folder, relative to
# My Drive, is the Colab Secret DRIVE_FOLDER (kept out of this public file).
MODE = "simulated"
REPO_URL = "https://github.com/DGK-Law-and-Cognition-Lab/Moral-Authority-in-Copyright.git"
CLONE = "/content/Moral-Authority-in-Copyright"

import os, sys, json, subprocess
assert MODE in ("simulated", "real")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    try:
        DRIVE_COPY = os.path.join("/content/drive/MyDrive", userdata.get("DRIVE_FOLDER").strip().strip("/"))
    except Exception as e:
        raise RuntimeError("Add the Colab Secret DRIVE_FOLDER (the Drive copy's folder, relative to My Drive) "
                           "and turn on this notebook's access to it.") from e
    if not os.path.isdir(os.path.join(DRIVE_COPY, "Analysis", "output")):
        raise FileNotFoundError("DRIVE_FOLDER does not point to a copy of the repository (no Analysis/output there).")
    if not os.path.exists(CLONE):
        subprocess.run(["git", "clone", "--quiet", REPO_URL, CLONE], check=True)
    os.chdir(CLONE)
sys.path.insert(0, os.path.join(os.getcwd() if IN_COLAB else os.path.abspath(".."), "python"))
sys.path.insert(0, os.path.abspath(os.path.join("Analysis", "python")))
import textlib as tl
ROOT = tl.repo_root()
if IN_COLAB:
    tl.link_drive_outputs(ROOT, os.path.join(DRIVE_COPY, "Analysis", "output"), [MODE])
CFG = tl.load_config(ROOT)
SEED = CFG["seed"]
COMMIT = subprocess.run(["git", "-C", ROOT, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository folder:", os.path.basename(ROOT), "| code commit:", COMMIT, "| mode:", MODE,
      "| text config frozen:", CFG["frozen"])


In [ ]:
if IN_COLAB and MODE == "real":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers", "google-genai", "voyageai", "bertopic", "umap-learn", "hdbscan", "pyarrow"], check=True)

In [ ]:
import numpy as np, pandas as pd
out = tl.output_dir(ROOT, MODE)
mt = pd.read_csv(os.path.join(out, "masked_texts.csv"), dtype={"pid": str}).fillna("")
ti = tl.load_text_input(ROOT, MODE).set_index("pid")
mt = mt.join(ti[["artist", "victim", "transform", "analyzed"]], on="pid")
print(len(mt), "texts")

## Embedding functions
In simulated mode every model is replaced by a hashing stand-in (`tl.hash_embed`), labeled as such in the output. It is not a language model.

In [ ]:
def get_secret(name):
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get(name)
    return os.environ.get(name)

def embed(model_cfg, texts):
    if MODE == "simulated":
        import hashlib
        return tl.hash_embed(texts, 256, seed=int(hashlib.sha256(model_cfg["key"].encode()).hexdigest(), 16) % 10000)
    p = model_cfg["provider"]
    if p == "sentence-transformers":
        import torch
        from sentence_transformers import SentenceTransformer
        if MODE == "real" and CFG["frozen"]:
            assert model_cfg.get("revision"), f"{model_cfg['id']}: revision must be pinned before the test set"
        m = SentenceTransformer(model_cfg["id"], revision=model_cfg.get("revision"),
                                model_kwargs={"torch_dtype": torch.bfloat16} if "Qwen" in model_cfg["id"] else {})
        return m.encode(list(texts), batch_size=16, normalize_embeddings=True, show_progress_bar=True)
    if p == "google":
        from google import genai
        from google.genai import types
        client = genai.Client(api_key=get_secret("GEMINI_API_KEY"))
        vecs = []
        for i in range(0, len(texts), 50):
            r = client.models.embed_content(model=model_cfg["id"], contents=list(texts[i:i + 50]),
                                            config=types.EmbedContentConfig(task_type=model_cfg["task_type"]))
            vecs += [e.values for e in r.embeddings]
        return tl.l2_normalize(np.array(vecs))
    if p == "voyage":
        import voyageai
        vo = voyageai.Client(api_key=get_secret("VOYAGE_API_KEY"))
        vecs = []
        for i in range(0, len(texts), 64):
            vecs += vo.embed(list(texts[i:i + 64]), model=model_cfg["id"], input_type=model_cfg["input_type"]).embeddings
        return tl.l2_normalize(np.array(vecs))
    raise ValueError(p)

In [ ]:
ecfg = CFG["embedding_models"]
models = [ecfg["primary"]] + ecfg["checks"]
emb_dir = tl.output_dir(ROOT, MODE, "embeddings")
E = {}
for m in models:
    for version in ("raw", "masked"):
        texts = mt["Explain_Ruling"] if version == "raw" else mt["masked"]
        X = np.asarray(embed(m, texts.tolist()), dtype=np.float32)
        E[(m["key"], version)] = X
        frame = pd.DataFrame(X, columns=[f"e{i}" for i in range(X.shape[1])])
        frame.insert(0, "pid", mt["pid"].values)
        frame.to_parquet(os.path.join(emb_dir, f"{m['key']}__{version}.parquet"), index=False)
        print(m["key"], version, X.shape, "(SIMULATED stand-in)" if MODE == "simulated" else "")

## Semantic axes (7.6)
Score = cos(response, mean of pole A anchors) - cos(response, mean of pole B anchors), on the masked text, for every model.

In [ ]:
anchors = tl.load_anchors(ROOT)
if MODE == "real" and CFG["frozen"]:
    assert anchors["frozen"], "anchors.json must be frozen before the test set"
rows = {"pid": mt["pid"].values}
for m in models:
    for ax, poles in anchors["axes"].items():
        A = np.asarray(embed(m, poles["pole_a"])); B = np.asarray(embed(m, poles["pole_b"]))
        rows[f"ddr_{ax}__{m['key']}"] = tl.ddr_scores(E[(m["key"], "masked")], A, B)
pd.DataFrame(rows).to_csv(os.path.join(out, "ddr_scores.csv"), index=False)
print("ddr_scores.csv written")

## Decoding classifier (7.4, effect size for T1-T3)
Test set only, masked text. Cross-validated L2 logistic regression (10 folds x 10 repeats) and a permutation test.

In [ ]:
dc = CFG["decoding"]
n_perm = dc["n_permutations"] if MODE == "real" else 100   # fewer permutations in simulated mode, for speed
test = (mt.text_set == "test") & mt.analyzed.astype(bool)
res = []
if test.sum() > 0:
    for m in models:
        X = E[(m["key"], "masked")][test.values]
        for f in ("artist", "victim", "transform"):
            y = mt.loc[test, f].values
            r = tl.decoding_accuracy(X, y, dc["folds"], dc["repeats"], n_perm, dc["C"], SEED)
            res.append({"model": m["key"], "factor": f, **r})
pd.DataFrame(res).to_csv(os.path.join(out, "decoding_T1_T3.csv"), index=False)
pd.DataFrame(res)

## BERTopic (7.6)
Primary embeddings of the masked text, all explanations; 10 seeds; stability as mean adjusted Rand index.

In [ ]:
bc = CFG["bertopic"]
X = E[(ecfg["primary"]["key"], "masked")]
assign = []
for s in bc["seeds"]:
    a, tm = tl.bertopic_assignments(mt["masked"].tolist(), X, s, bc["min_topic_size"])
    assign.append(a)
    if s == bc["seeds"][0]:
        tm.get_topic_info().to_csv(os.path.join(out, "bertopic_topic_info.csv"), index=False)
pd.DataFrame({"pid": mt["pid"], **{f"seed_{s}": a for s, a in zip(bc["seeds"], assign)}}).to_csv(
    os.path.join(out, "bertopic_assignments.csv"), index=False)
stab = tl.topic_stability(assign)
json.dump(stab, open(os.path.join(out, "bertopic_stability.json"), "w"), indent=2)
stab